# 观测埋点

**常见用法**：耗时统计（找慢工具）、审计日志（谁/何时/什么参数）、成败率监控
（失败率突增 = 下游异常）、结果长度告警（防撑爆上下文）——一处覆盖全部工具调用。

**钩子内的做法**：
- `time.perf_counter()` 包裹 `execute` 计时；thread_id/角色从 `request.runtime.config["configurable"]` 取
- 成败看 `result.status`（错误已被 handle_tool_errors 包成 error ToolMessage）；异常型错误用 try/except 计数后**原样 raise**
- 纪律：只记录，不改行为——吞异常会连 `GraphInterrupt` 一起吞掉，审批直接失效

##  1. 耗时统计

In [22]:

from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    time.sleep(5)
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    time.sleep(3)
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


def timed(request: ToolCallRequest, execute) -> ToolMessage | object:
    """耗时统计"""
    t0 = time.perf_counter()
    result = execute(request)
    cost = (time.perf_counter() - t0) * 1000
    print(f"[耗时] {request.tool_call['name']}: {cost:.0f}ms")
    return result


# 工具节点：钩子接管审批，错误按 handler 策略处理
tool_node = ToolNode(tools, wrap_tool_call=timed)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: ChatState) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}
graph = builder.compile(checkpointer)
res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气,以及科技和体育方面的新闻")]}, config=config)
print(res)

[耗时] get_news: 3001ms

[耗时] get_news: 3026ms

[耗时] get_weather: 5001ms

{
    'messages': [
        HumanMessage(
            content='帮我查一下北京的天气,以及科技和体育方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='8c3ef8a3-e707-41c4-bfc4-7f184b27580d'
        ),
        AIMessage(
            content='我来帮您查询北京的天气以及科技和体育新闻。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 107,
                    'prompt_tokens': 348,
                    'total_tokens': 455,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 220
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '67d3d5a5-2c21-4661-aa8b-4b4b0ccc67e3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0ad-8ee5-79f3-bd8e-662517e404cc-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_TJ1ZDKyxmusjf4C1Vm312543',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_ILcBtTZM7e30jhT5UdSt7600',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '体育'},
                    'id': 'call_02_DWXqYeWlOiYaS7QpThbq5225',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 348,
                'output_tokens': 107,
                'total_tokens': 455,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='北京 的天气是晴天，温度 25°C',
            name='get_weather',
            id='cbd42680-84be-448b-aba2-8719b7085638',
            tool_call_id='call_00_TJ1ZDKyxmusjf4C1Vm312543'
        ),
        ToolMessage(
            content='最新科技新闻：AI 技术正在快速发展。',
            name='get_news',
            id='425b5b06-f5c7-42af-a785-54587604ed8e',
            tool_call_id='call_01_ILcBtTZM7e30jhT5UdSt7600'
        ),
        ToolMessage(
            content='最新体育新闻：中国队取得了胜利。',
            name='get_news',
            id='9aba146d-d43a-4828-82bc-f3dfe06e58ed',
            tool_call_id='call_02_DWXqYeWlOiYaS7QpThbq5225'
        ),
        AIMessage(
            content='已经帮您查询好了，以下是结果：\n\n🌤️ **北京天气**\n- 天气：晴天\n- 温度：25°C\n- 
温馨提示：天气晴朗，温度适宜，适合外出活动，记得做好防晒哦～\n\n📱 **科技新闻**\n- 最新科技新闻：AI 
技术正在快速发展。\n\n⚽ **体育新闻**\n- 
最新体育新闻：中国队取得了胜利。\n\n如果您还想了解其他城市天气或其他方面的新闻（如娱乐），随时告诉我！',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 99,
                    'prompt_tokens': 511,
                    'total_tokens': 610,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_mi

## 2. 调用工具审计

In [21]:

from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


def audit(request: ToolCallRequest, execute) -> ToolMessage | object:
    """工具审计"""
    tc = request.tool_call
    thread_id = request.runtime.config["configurable"]["thread_id"]
    print(f"[审计] thread_id: {thread_id}, {tc['name']} : {tc['args']}")
    res = execute(request)
    print(f"[审计] thread_id: {thread_id}, {tc['name']} : {res.status}")
    return res


# 工具节点：钩子接管审批，错误按 handler 策略处理
tool_node = ToolNode(tools, wrap_tool_call=audit)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: ChatState) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}
graph = builder.compile(checkpointer)
res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气,以及科技和体育方面的新闻")]}, config=config)
print(res)

[审计] thread_id: 1, get_weather : {'city': '北京'}

[审计] thread_id: 1, get_weather : success

[审计] thread_id: 1, get_news : {'topic': '体育'}

[审计] thread_id: 1, get_news : {'topic': '科技'}

[审计] thread_id: 1, get_news : success

[审计] thread_id: 1, get_news : success

{
    'messages': [
        HumanMessage(
            content='帮我查一下北京的天气,以及科技和体育方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='2d96732b-d41f-493f-873e-f58e9df16bc1'
        ),
        AIMessage(
            content='我来帮您查询北京的天气以及科技和体育新闻。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 107,
                    'prompt_tokens': 348,
                    'total_tokens': 455,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 220
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'ed031e85-e049-4f2e-a991-6e5a88703773',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0ad-6db2-7320-8f59-ccf2f3f21755-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_vZf9srWCFkjqrpEa2dbj1396',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_XhY8ibqmIJpurhGw9SET6446',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '体育'},
                    'id': 'call_02_DCsB7J7iwZPmEPgFa5EN3011',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 348,
                'output_tokens': 107,
                'total_tokens': 455,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='北京 的天气是晴天，温度 25°C',
            name='get_weather',
            id='b035fc8b-a398-4b91-ada9-afc956ece493',
            tool_call_id='call_00_vZf9srWCFkjqrpEa2dbj1396'
        ),
        ToolMessage(
            content='最新科技新闻：AI 技术正在快速发展。',
            name='get_news',
            id='d07fd1f4-9b0f-455f-b14b-def125117b4c',
            tool_call_id='call_01_XhY8ibqmIJpurhGw9SET6446'
        ),
        ToolMessage(
            content='最新体育新闻：中国队取得了胜利。',
            name='get_news',
            id='833b6588-e0f5-4a1c-b0b7-fbb4e0418b4a',
            tool_call_id='call_02_DCsB7J7iwZPmEPgFa5EN3011'
        ),
        AIMessage(
            content='为您查询到以下信息：\n\n**🌤️ 北京天气**\n- 天气：晴天\n- 温度：25°C\n\n**💻 科技新闻**\n- 
最新科技新闻：AI 技术正在快速发展。\n\n**⚽ 体育新闻**\n- 
最新体育新闻：中国队取得了胜利。\n\n北京今天天气晴朗舒适（25°C），适合外出活动。需要我帮您查询其他城市或其他方面的
信息吗？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 511,
                    'total_tokens': 601,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 255
           

## 3. 成败统计

In [19]:
import time
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    raise ConnectionError("连接超时")
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


result_dict: dict[str, dict[str, int]] = {}


def count_result(request: ToolCallRequest, execute) -> ToolMessage | object:
    """成败统计：{工具名: {status: 次数}}"""
    tc = request.tool_call
    res = execute(request)
    counts = result_dict.setdefault(tc["name"], {})  # 没有该工具就建空 dict
    counts[res.status] = counts.get(res.status, 0) + 1
    return res


# 工具节点
tool_node = ToolNode(tools, wrap_tool_call=count_result, handle_tool_errors=True)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: ChatState) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}
graph = builder.compile(checkpointer)
res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气,以及科技和体育方面的新闻")]}, config=config)
print(res)

{
    'messages': [
        HumanMessage(
            content='帮我查一下北京的天气,以及科技和体育方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='55ffc176-dabf-49b6-96c2-de4723d59d2c'
        ),
        AIMessage(
            content="I'll fetch all three pieces of information for you in parallel.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 108,
                    'prompt_tokens': 348,
                    'total_tokens': 456,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 220
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'bca8d258-9702-444c-ba29-2f536ee0bbb3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0ab-1d9d-7772-9c8e-ef51d5b8c6b4-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_xpt1XCW7lwIuiKseI5Wx4808',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_dK2ktr3uLIJRZ85lfqjE3415',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '体育'},
                    'id': 'call_02_zPlDQBLr5AIMuXAn5nGO9608',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 348,
                'output_tokens': 108,
                'total_tokens': 456,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Error: ConnectionError('连接超时')\n Please fix your mistakes.",
            name='get_weather',
            id='8e23527a-f57e-4b84-8dc4-a9eadce2cca8',
            tool_call_id='call_00_xpt1XCW7lwIuiKseI5Wx4808',
            status='error'
        ),
        ToolMessage(
            content='最新科技新闻：AI 技术正在快速发展。',
            name='get_news',
            id='32192fbe-49f5-4ed3-83d0-99f6643e0f1e',
            tool_call_id='call_01_dK2ktr3uLIJRZ85lfqjE3415'
        ),
        ToolMessage(
            content='最新体育新闻：中国队取得了胜利。',
            name='get_news',
            id='2f9f3390-52e1-4b3d-bb15-b7a8394ecb27',
            tool_call_id='call_02_zPlDQBLr5AIMuXAn5nGO9608'
        ),
        AIMessage(
            content='这是查询结果：\n\n**🏙️ 
北京天气**\n查询失败了，返回的是「连接超时」错误。天气服务暂时不可用，建议稍后重试。\n\n**💻 
科技新闻**\n最新科技新闻：AI 技术正在快速发展。\n\n**⚽ 
体育新闻**\n最新体育新闻：中国队取得了胜利。\n\n---\n\n需要我重新查一次北京的天气吗？或者换其他城市、其他新闻主题也
可以。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 513,
                    'total_tokens': 601,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    

In [20]:
print(result_dict)

{'get_weather': {'error': 1}, 'get_news': {'success': 2}}